[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/69_cycle_flip_maintenance_solution.ipynb)

# Solution: Single-Cycle Graph Alarm Flip

Reference solution — spanning tree + leaf-to-root greedy, then flip the unique cycle.


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
from typing import List


In [ ]:
# ✅ SOLUTION

class Solution:
    def solve(self, n: int, state: List[int], edges: List[List[int]]) -> List[int]:
        E = [(u - 1, v - 1, c) for (u, v, c) in edges]
        m = len(E)
        graph = [[] for _ in range(n)]
        for i, (u, v, c) in enumerate(E):
            graph[u].append((v, i))
            graph[v].append((u, i))
        parent = [-1] * n
        parent_edge = [-1] * n
        depth = [0] * n
        tree_edge = [False] * m
        order = []
        parent[0] = 0
        stack = [0]
        while stack:                          # iterative DFS spanning tree
            u = stack.pop()
            order.append(u)
            for v, eid in graph[u]:
                if parent[v] == -1:
                    parent[v] = u
                    parent_edge[v] = eid
                    depth[v] = depth[u] + 1
                    tree_edge[eid] = True
                    stack.append(v)
        extra = next(i for i in range(m) if not tree_edge[i])  # the one non-tree edge
        # leaf -> root: fix each node using its parent edge
        selected = [False] * m
        need = list(state)
        for u in reversed(order[1:]):
            if need[u] == 1:
                selected[parent_edge[u]] = True
                need[parent[u]] ^= 1
        if need[0] == 1:                       # root still on -> parity odd -> infeasible
            return [-1, 0]
        first = sum(E[i][2] for i in range(m) if selected[i])
        # mark the unique cycle: non-tree edge + tree path between its endpoints
        on_cycle = [False] * m
        on_cycle[extra] = True
        u, v, _ = E[extra]
        while depth[u] > depth[v]:
            on_cycle[parent_edge[u]] = True; u = parent[u]
        while depth[v] > depth[u]:
            on_cycle[parent_edge[v]] = True; v = parent[v]
        while u != v:
            on_cycle[parent_edge[u]] = True; on_cycle[parent_edge[v]] = True
            u = parent[u]; v = parent[v]
        # the other feasible plan = toggle every cycle edge
        second = first
        for i in range(m):
            if on_cycle[i]:
                second += -E[i][2] if selected[i] else E[i][2]
        if first == second:
            return [first, 2]
        return [min(first, second), 1]


In [ ]:
# Demo
sol = Solution()
print(sol.solve(5, [0, 1, 1, 1, 1], [[1, 2, 4], [2, 3, 2], [3, 1, 7], [3, 4, 5], [4, 5, 1]]))  # [3, 1]
print(sol.solve(4, [1, 1, 1, 1], [[1, 2, 1], [2, 3, 1], [3, 4, 1], [4, 1, 1]]))                # [2, 2]
print(sol.solve(3, [1, 0, 0], [[1, 2, 1], [2, 3, 1], [3, 1, 1]]))                              # [-1, 0]


In [ ]:
from torch_judge import check
check('cycle_flip_maintenance')
